In [718]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

In [719]:
# Plot
plot_directory = 'result/plot_set'

In [720]:
colors = [
	'#FF0000',
	'#00FFFF',
	'#0000FF',
	'#00008B',
	'#ADD8E6',
	'#800080',
	'#7FFFD4',
	'#008000',
	'#FF00FF',
	'#FFC0CB',
	'#C0C0C0',
	'#FFA500',
	'#000000',
	'#800000',
]

In [721]:
def load_json_file(file_path):
	try:
		with open(file_path, 'r') as file:
			data = json.load(file)
		return data
	
	except Exception as e:
		print(f"An error occurred while loading the JSON file: {e}")
		return None

In [722]:
def extract_fpss(metric_list):
	return list(metric_list[list(metric_list.keys())[0]][0]['metric'].keys())

In [723]:
def to_accuracy_vector(accuracy_result_seq, fpss, type='F1'):
	accuracy_vector = []
	for fps in fpss:
		accuracy_vector.append(accuracy_result_seq[fps][type])
	
	return accuracy_vector

In [724]:
def plot_scatter(xs, ys, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	plt.scatter(xs, ys, c=colors[0], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [725]:
def plot_scatter_label(xs, ys, labels, x_label, y_label, title, fig_size=(8, 8)):
	plt.figure(figsize=fig_size)
	for i in range(len(xs)):
		plt.scatter(xs[i], ys[i], c=colors[labels[i]], s=30, alpha=0.5)
	plt.title(title)
	plt.xlabel(x_label)
	plt.ylabel(y_label)
	plt.show()

In [726]:
def filter_list_index(input_list, indices_to_remove):
	return [item for i, item in enumerate(input_list) if i not in indices_to_remove]

In [727]:
def round_float_to_sigfigs(number, sigfigs):
	return round(number, sigfigs)

## Plot

In [728]:
omv_features = ["Left-Top", "Right-Top", "Left-Bottom", "Right-Bottom", "Object-Amount", "Confidence", "IOU"]

In [729]:
plot_filenames = sorted(os.listdir(plot_directory))
plot_video_names = sorted(list(set([f.split('_')[0] for f in plot_filenames])))

In [730]:
fpss = extract_fpss(load_json_file(os.path.join(plot_directory, plot_video_names[0] + "_Accuracy_Result.json")))

In [731]:
omv_videos = []
acc_videos = []

for v in plot_video_names:
	omv_dict = {}
	for fps in fpss:
		omv_dict[fps] = []
	acc_list = []

	accuracy_result = load_json_file(os.path.join(plot_directory, v + "_Accuracy_Result.json"))
	movement_result = load_json_file(os.path.join(plot_directory, v + "_Movement_Result.json"))

	for class_idx in list(accuracy_result.keys()):
		for i in range(len(accuracy_result[class_idx])):
			accuracy_vector = to_accuracy_vector(accuracy_result[class_idx][i]['metric'], fpss)
			acc_list.append(accuracy_vector)

			for fps in fpss:
				movement_vector = movement_result[class_idx][i]['movement'][fps]
				omv_dict[fps].append(movement_vector)
	
	omv_videos.append(omv_dict)
	acc_videos.append(acc_list)

In [732]:
# Corr

# for i in range(len(plot_video_names)):
# 	video_name = plot_video_names[i]
# 	for k in range(len(fpss)):
# 		for j in range(len(omv_videos[0][fpss[0]][0])):	
# 			fps = fpss[k]

# 			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
# 			acc_fps = list(np.array(acc_videos[i])[:, k])

# 			# Remove Outliers
# 			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
# 			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
# 			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

# 			title = f'{video_name}; OMV Feature {j} ({omv_features[j]}); FPS: {fps}'

# 			correlation_matrix = np.corrcoef(omv_fps_clean, acc_fps_clean)
# 			correlation_coefficient = correlation_matrix[0, 1]
# 			print(f"{title} -> Corr: {round_float_to_sigfigs(correlation_coefficient, 3)}")
# 			# plot_scatter(omv_fps_clean, acc_fps_clean, 'OMV Feature', 'ACC', title)

# 		print("")

# New Code

In [733]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures

In [734]:
def transpose_array(arr):
    # Transpose the input array using zip and map it back to a list
    return [list(row) for row in zip(*arr)]

In [735]:
def my_train_test_split(x, y, test_size):
	split_point = int(len(x) * (1-test_size))
	return x[0:split_point], x[split_point:len(x)], y[0:split_point], y[split_point:len(x)]

In [736]:
OMV_FEATURE_INDEX = [4, 6]
TEST_SIZE = 0.2

In [737]:
# Dataset Build

dataset = {}

for i in range(len(plot_video_names)):
	video_name = plot_video_names[i]
	dataset[video_name] = {}
	
	for k in range(len(fpss)):
		dataset[video_name][k] = {}
		y_all = None
		x_all_t = []

		for j in range(len(omv_videos[0][fpss[0]][0])):	
			fps = fpss[k]

			omv_fps = list(np.array(omv_videos[i][fps])[:, j])
			acc_fps = list(np.array(acc_videos[i])[:, k])

			# Remove Outliers
			outlier_index = [l for l in range(len(omv_fps)) if omv_fps[l] == -1]
			omv_fps_clean = filter_list_index(omv_fps, outlier_index)
			acc_fps_clean = filter_list_index(acc_fps, outlier_index)

			if j == 0:
				y_all = acc_fps_clean.copy()
			if j in OMV_FEATURE_INDEX:
				x_all_t.append(omv_fps_clean.copy())
		
		x_all = transpose_array(x_all_t)
		dataset[video_name][k]['y_all'] = y_all
		dataset[video_name][k]['x_all'] = x_all
			

In [738]:
# for i in range(len(plot_video_names)):
# 	video_name = plot_video_names[i]
# 	dataset[video_name] = {}
	
# 	for k in range(len(fpss)):

In [739]:
x_all = dataset['Video1'][0]['x_all']
y_all = dataset['Video1'][0]['y_all']

x_train, x_test, y_train, y_test = my_train_test_split(x_all, y_all, TEST_SIZE)

In [740]:
# LR
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)

y_train_pred = lr_model.predict(x_train)
y_test_pred = lr_model.predict(x_test)

train_mse = mean_squared_error(y_train, y_train_pred)
print(f"Linear Regression Train MSE: {round(train_mse, 4)}")

test_mse = mean_squared_error(y_test, y_test_pred)
print(f"Linear Regression Test MSE: {round(test_mse, 4)}")

Linear Regression Train MSE: 0.0204
Linear Regression Test MSE: 0.0123


In [741]:
# PR
poly_train = PolynomialFeatures(degree=3)
poly_test = PolynomialFeatures(degree=3)
x_train_poly = poly_train.fit_transform(x_train)
x_test_poly = poly_test.fit_transform(x_test)

pr_model = LinearRegression()
pr_model.fit(x_train_poly, y_train)

y_train_pred = pr_model.predict(x_train_poly)
y_test_pred = pr_model.predict(x_test_poly)

train_mse = mean_squared_error(y_train, y_train_pred)
print(f"Polynomial Regression Train MSE: {round(train_mse, 4)}")

test_mse = mean_squared_error(y_test, y_test_pred)
print(f"Polynomial Regression Test MSE: {round(test_mse, 4)}")

Polynomial Regression Train MSE: 0.019
Polynomial Regression Test MSE: 0.0111


In [742]:
# RFR
rfr_model = RandomForestRegressor(n_estimators=100, random_state=42)
rfr_model.fit(x_train, y_train)

y_train_pred = rfr_model.predict(x_train)
y_test_pred = rfr_model.predict(x_test)

train_mse = mean_squared_error(y_train, y_train_pred)
print(f"Random Forest Regression Train MSE: {round(train_mse, 4)}")

test_mse = mean_squared_error(y_test, y_test_pred)
print(f"Random Forest Regression Test MSE: {round(test_mse, 4)}")

Random Forest Regression Train MSE: 0.0032
Random Forest Regression Test MSE: 0.0116


In [743]:
# SVR
svr_model = SVR(kernel='rbf', C=1e3, gamma=0.1)
svr_model.fit(x_train, y_train)

y_train_pred = svr_model.predict(x_train)
y_test_pred = svr_model.predict(x_test)

train_mse = mean_squared_error(y_train, y_train_pred)
print(f"Support Vector Regression Train MSE: {round(train_mse, 4)}")

test_mse = mean_squared_error(y_test, y_test_pred)
print(f"Support Vector Regression Test MSE: {round(test_mse, 4)}")

Support Vector Regression Train MSE: 0.0175
Support Vector Regression Test MSE: 0.013


In [744]:
# KNN
knn_model = KNeighborsRegressor(n_neighbors=5)  # Using k = 5 neighbors
knn_model.fit(x_train, y_train)

y_train_pred = knn_model.predict(x_train)
y_test_pred = knn_model.predict(x_test)

train_mse = mean_squared_error(y_train, y_train_pred)
print(f"KNN Train MSE: {round(train_mse, 4)}")

test_mse = mean_squared_error(y_test, y_test_pred)
print(f"KNN Test MSE: {round(test_mse, 4)}")

KNN Train MSE: 0.0126
KNN Test MSE: 0.0149
